In [ ]:
import os
from google.cloud import bigquery
from google.oauth2 import service_account
from dotenv import load_dotenv

load_dotenv()

project_id = "mi-proyecto-net-510916"
dataset_id = "tu_empresa"
credentials_path = "../../credentials/service-account.json"

credentials = service_account.Credentials.from_service_account_file(credentials_path)

client = bigquery.Client(credentials=credentials, project=project_id)

dataset_ref = f"{project_id}.{dataset_id}"
dataset = bigquery.Dataset(dataset_ref)
dataset.location = "EU"
try:
    dataset = client.create_dataset(dataset, exists_ok=True)
    print(f"Dataset {dataset_id} creado o verificado correctamente en Europa.")
except Exception as e:
    print(f"Error al crear el dataset: {e}")

queries = [
    f"""
    CREATE TABLE IF NOT EXISTS `{dataset_ref}.customers` (
        customer_id STRING OPTIONS(description="Identificador único del cliente"),
        first_name STRING,
        last_name STRING,
        email STRING,
        country STRING,
        city STRING,
        acquisition_channel STRING,
        created_at TIMESTAMP
    );
    """,
    f"""
    CREATE TABLE IF NOT EXISTS `{dataset_ref}.categories` (
        category_id STRING OPTIONS(description="Identificador único de la categoría"),
        name STRING,
        description STRING
    );
    """,
    f"""
    CREATE TABLE IF NOT EXISTS `{dataset_ref}.products` (
        product_id STRING OPTIONS(description="Identificador único del producto"),
        name STRING,
        category_id STRING,
        price FLOAT64,
        cost FLOAT64,
        stock INT64,
        is_active BOOL
    );
    """,
    f"""
    CREATE TABLE IF NOT EXISTS `{dataset_ref}.orders` (
        order_id STRING OPTIONS(description="Identificador único del pedido"),
        customer_id STRING,
        status STRING,
        shipping_address STRING,
        shipping_country STRING,
        order_date TIMESTAMP,
        shipped_date TIMESTAMP,
        delivered_date TIMESTAMP
    );
    """,
    f"""
    CREATE TABLE IF NOT EXISTS `{dataset_ref}.order_items` (
        order_item_id STRING OPTIONS(description="Identificador único de la línea de pedido"),
        order_id STRING,
        product_id STRING,
        quantity INT64,
        unit_price FLOAT64,
        discount FLOAT64
    );
    """,
    f"""
    CREATE TABLE IF NOT EXISTS `{dataset_ref}.payments` (
        payment_id STRING OPTIONS(description="Identificador único del pago"),
        order_id STRING,
        payment_method STRING,
        status STRING,
        amount FLOAT64,
        payment_date TIMESTAMP
    );
    """,
    f"""
    CREATE TABLE IF NOT EXISTS `{dataset_ref}.reviews` (
        review_id STRING OPTIONS(description="Identificador único de la valoración"),
        order_item_id STRING,
        customer_id STRING,
        rating INT64,
        comment STRING,
        created_at TIMESTAMP
    );
    """
]

# Ejecutar la creación de cada tabla
for i, query in enumerate(queries, 1):
    query_job = client.query(query)
    query_job.result()
    print(f"Tabla {i} creada correctamente en BigQuery.")

TypeError: str expected, not NoneType